In [9]:
# =====================================================
# Clean install (Python 3.12 compatible)
# =====================================================

!pip uninstall -y transformers sentence-transformers tokenizers huggingface-hub faiss-cpu -q

!pip install \
transformers==4.45.2 \
sentence-transformers==3.1.1 \
huggingface-hub==0.25.2 \
tokenizers==0.20.1 \
faiss-cpu \
-q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.38.0 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.25.2 which is incompatible.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.25.2 which is incompatible.


In [3]:
# =====================================================
# Install
# =====================================================
!pip install faiss-cpu sentence-transformers transformers -q


# =====================================================
# Imports
# =====================================================
import pandas as pd
import numpy as np
import faiss
import torch

from sentence_transformers import (
    SentenceTransformer,
    CrossEncoder
)

from transformers import (
    AutoTokenizer,
    pipeline
)


# =====================================================
# Load data
# =====================================================
train = pd.read_csv("train.csv")

print(train.shape)
print(train.head())


# =====================================================
# Build Knowledge Base
# =====================================================
kb=[]

for idx,row in train.iterrows():

    correct_letter=row["answer"]

    kb.append(
        str(row[correct_letter])
    )


print("KB size:",len(kb))


# =====================================================
# Embedding model + FAISS
# =====================================================
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)


kb_embeddings = model.encode(
    kb,
    show_progress_bar=True
)


index = faiss.IndexFlatL2(
    kb_embeddings.shape[1]
)

index.add(kb_embeddings)


print("FAISS ready")

(2000, 8)
   id                                             prompt  \
0   1  Pick the best possible answer: What is Martin ...   
1   2        What is accelerator-based light-ion fusion?   
2   3  Determine the correct option: What is the term...   
3   4  Select the most accurate option: What is Marti...   
4   5  Identify the correct statement: What is the co...   

                                                   A  \
0  Martin Heidegger believes that humans exist wi...   
1  Accelerator-based light-ion fusion is a techni...   
2                                       Blueshifting   
3  Martin Heidegger believes that humans exist wi...   
4  Simultaneity is relative, meaning that two eve...   

                                                   B  \
0  Martin Heidegger believes that humans do not e...   
1  Accelerator-based light-ion fusion is a techni...   
2                                        Redshifting   
3  Martin Heidegger believes that humans do not e...   
4  Simultane

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:90: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

FAISS ready


In [1]:
import transformers
import sentence_transformers
import faiss
import huggingface_hub

print("transformers:", transformers.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("faiss loaded")
print("hub:", huggingface_hub.__version__)

transformers: 4.45.2
sentence-transformers: 3.1.1
faiss loaded
hub: 0.25.2


In [4]:
zs = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)


row_150=train.iloc[150]


prompt_150=str(row_150["prompt"])


labels_150=[
    str(row_150[x])
    for x in ["A","B","C","D","E"]
]


correct_text=str(
    row_150[row_150["answer"]]
)


result = zs(
    prompt_150,
    labels_150
)


score_dict=dict(
    zip(
        result["labels"],
        result["scores"]
    )
)


q1 = score_dict[correct_text]


print(
    "Q1 =",
    round(q1,3)
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


Q1 = 0.384


In [6]:
query_emb = model.encode(
    [prompt_150]
)


D,I=index.search(
    query_emb,
    10
)


retrieved_indices=I[0]


print(
    retrieved_indices
)


q2=list(
    retrieved_indices
).index(150)+1


print(
    "Q2 Rank =",
    q2
)

[ 663 1701 1269 1532  576  847 1693 1906  168  150]
Q2 Rank = 10


In [7]:
cross_encoder = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)


docs_10=[
    kb[i]
    for i in retrieved_indices
]


pairs=[
    [prompt_150,doc]
    for doc in docs_10
]


scores = cross_encoder.predict(
    pairs
)


reranked=np.argsort(
    scores
)[::-1]


new_order=[
    retrieved_indices[i]
    for i in reranked
]


q3=new_order.index(150)+1


print(
    "Q3 Rank =",
    q3
)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Q3 Rank = 1


In [8]:
tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)


row42=train.iloc[42]


emb42=model.encode(
    [row42["prompt"]]
)


D,I=index.search(
    emb42,
    5
)


docs=[
    kb[i]
    for i in I[0]
]


context=" ".join(docs)


rag_string = (
    "Context: "
    +context+
    " Question: "
    +row42["prompt"]
)


tokens=tokenizer(
    rag_string,
    truncation=False
)


q4=len(tokens["input_ids"])


print(
    "Q4 Tokens =",
    q4
)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Q4 Tokens = 216


In [9]:
true_doc=kb[150]


rag_true = (
    "Context: "
    +true_doc+
    " Question: "
    +prompt_150
)


result = zs(
    rag_true,
    labels_150
)


scores=dict(
    zip(
        result["labels"],
        result["scores"]
    )
)


q5=scores[correct_text]


print(
    "Q5 =",
    round(q5,3)
)

Q5 = 0.989


In [10]:
bad_doc=kb[999]


bad_rag = (
    "Context: "
    +bad_doc+
    " Question: "
    +prompt_150
)


result = zs(
    bad_rag,
    labels_150
)


scores=dict(
    zip(
        result["labels"],
        result["scores"]
    )
)


q6=scores[correct_text]


print(
    "Q6 =",
    round(q6,3)
)

Q6 = 0.529


In [11]:
hits=0


for i in range(100):

    row=train.iloc[i]

    prompt=row["prompt"]

    correct=str(
        row[row["answer"]]
    )


    emb=model.encode(
        [prompt]
    )


    D,I=index.search(
        emb,
        5
    )


    retrieved=[
        kb[x]
        for x in I[0]
    ]


    found=False


    for doc in retrieved:

        if correct in doc:
            found=True


    if found:
        hits+=1



hit_rate=hits/100*100


print(
    "Q7 Hit Rate =",
    round(hit_rate,1)
)

Q7 Hit Rate = 73.0


In [12]:
def apk(actual,pred,k=3):

    pred=pred[:k]

    if actual in pred:
        return 1/(pred.index(actual)+1)

    return 0



scores=[]


for i in range(20):

    row=train.iloc[i]


    prompt=str(row["prompt"])


    # retrieve

    emb=model.encode(
        [prompt]
    )


    D,I=index.search(
        emb,
        5
    )


    docs=[
        kb[x]
        for x in I[0]
    ]


    # rerank

    pairs=[
        [prompt,d]
        for d in docs
    ]


    ce_scores=cross_encoder.predict(
        pairs
    )


    best_doc=docs[
        np.argmax(ce_scores)
    ]


    # augment

    rag=(
        "Context: "
        +best_doc+
        " Question: "
        +prompt
    )


    option_letters=[
        "A","B","C","D","E"
    ]


    labels=[
        str(row[x])
        for x in option_letters
    ]


    result=zs(
        rag,
        labels
    )


    label_score=dict(
        zip(
            result["labels"],
            result["scores"]
        )
    )


    ranked_texts=sorted(
        label_score,
        key=label_score.get,
        reverse=True
    )


    ranked_letters=[]


    for txt in ranked_texts:

        ranked_letters.append(
            option_letters[
                labels.index(txt)
            ]
        )


    scores.append(
        apk(
            row["answer"],
            ranked_letters,
            3
        )
    )


q8=np.mean(scores)


print(
    "Q8 MAP@3 =",
    round(q8,3)
)

Q8 MAP@3 = 0.975
